# TVL Dashboard Readiness

Notebook de QA analytique pour verifier que les marts TVL sont coherents avant de brancher Metabase.

Objectifs:
- verifier la fraicheur et la couverture des tables TVL
- comparer `mart_tvl_current`, `mart_tvl_block_snapshot` et `mart_tvl_daily`
- verifier que `active + out_of_range = total`
- verifier que `mart_tvl_distribution` recompose bien la TVL courante
- inspecter le bin courant et la distribution autour du prix

## Prerequis

- `clickhouse_local` doit tourner sur `localhost:8124`
- les marts doivent etre build en local
- si tu veux controler un autre snapshot block-level, rebuild d'abord:

```bash
cd /home/gharbi/defind-analytics/dbt/defind_analytics
../../.venv/bin/dbt build --target local --select mart_tvl_block_snapshot --vars '{"target_block": 24820333}'
```

In [1]:
import os
from textwrap import dedent

import clickhouse_connect

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'pandas is required. Run: %pip install pandas clickhouse-connect plotly jupyterlab'
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ModuleNotFoundError:
    PLOTLY_AVAILABLE = False
    print('Plotly not available. Run: %pip install plotly')

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)

In [2]:
# Defaults local-first pour clickhouse_local; surcharge possible via variables d'environnement.
CLICKHOUSE_HOST = os.getenv('CLICKHOUSE_HOST', 'localhost')
CLICKHOUSE_PORT = int(os.getenv('CLICKHOUSE_PORT', '8124'))
CLICKHOUSE_USER = os.getenv('CLICKHOUSE_USER', 'default')
CLICKHOUSE_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '1234')
CLICKHOUSE_DATABASE = os.getenv('CLICKHOUSE_DATABASE', 'analytics')

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f'Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}')

Connected to ClickHouse database=analytics on localhost:8124


In [3]:
TIME_COLUMNS = ['date', 'snapshot_block_time', 'block_time', 'deployment_ts']

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    return df

def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    df = normalize_df(df)
    print(f'{len(df):,} rows')
    return df

def query_scalar(sql: str, parameters: dict | None = None):
    result = client.query(sql, parameters=parameters or {})
    return result.first_row[0] if result.result_rows else None

## 1. Pool selection and model freshness

In [4]:
available_pools = query_df(dedent(
    '''
    SELECT pool
    FROM mart_tvl_current
    ORDER BY pool
    '''
))
available_pools

1 rows


,pool
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640


In [5]:
POOL = os.getenv('POOL_ADDRESS', available_pools.iloc[0]['pool'] if not available_pools.empty else None)
assert POOL, 'No pool found in mart_tvl_current.'
print(f'Using pool: {POOL}')

Using pool: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640


In [6]:
tvl_inventory = query_df(dedent(
    '''
    SELECT
        table_name,
        rows,
        min_time,
        max_time,
        max_block_or_target
    FROM (
        SELECT
            'stg_swaps' AS table_name,
            count() AS rows,
            min(block_time) AS min_time,
            max(block_time) AS max_time,
            toUInt64(max(block_number)) AS max_block_or_target
        FROM stg_swaps
        WHERE pool = %(pool)s

        UNION ALL

        SELECT
            'mart_tvl_daily' AS table_name,
            count() AS rows,
            min(toDateTime(date)) AS min_time,
            max(toDateTime(date)) AS max_time,
            toUInt64(0) AS max_block_or_target
        FROM mart_tvl_daily
        WHERE pool = %(pool)s

        UNION ALL

        SELECT
            'mart_tvl_current' AS table_name,
            count() AS rows,
            NULL AS min_time,
            NULL AS max_time,
            toUInt64(0) AS max_block_or_target
        FROM mart_tvl_current
        WHERE pool = %(pool)s

        UNION ALL

        SELECT
            'mart_tvl_block_snapshot' AS table_name,
            count() AS rows,
            min(snapshot_block_time) AS min_time,
            max(snapshot_block_time) AS max_time,
            max(target_block) AS max_block_or_target
        FROM mart_tvl_block_snapshot
        WHERE pool = %(pool)s

        UNION ALL

        SELECT
            'mart_tvl_distribution' AS table_name,
            count() AS rows,
            NULL AS min_time,
            NULL AS max_time,
            toUInt64(0) AS max_block_or_target
        FROM mart_tvl_distribution
        WHERE pool = %(pool)s
    )
    ORDER BY table_name
    '''
), {'pool': POOL})
tvl_inventory

5 rows


,table_name,rows,min_time,max_time,max_block_or_target
0,mart_tvl_block_snapshot,1,2026-04-06 11:28:11,2026-04-06 11:28:11,24820333
1,mart_tvl_current,1,NaT,NaT,0
2,mart_tvl_daily,1798,2021-05-05 00:00:00,2026-04-06 00:00:00,0
3,mart_tvl_distribution,1537,NaT,NaT,0
4,stg_swaps,10776354,2021-05-05 22:15:01,2026-04-06 11:28:11,24820333


## 2. Current snapshot vs block snapshot vs latest daily close

Le but ici est de verifier que:
- `mart_tvl_current` et `mart_tvl_block_snapshot` sont alignes quand le block snapshot pointe sur le dernier block construit
- `mart_tvl_daily` est coherent avec le dernier close journalier disponible

In [7]:
snapshot_compare = query_df(dedent(
    '''
    WITH latest_daily AS (
        SELECT
            pool,
            date AS latest_daily_date,
            price_token1_in_token0 AS latest_daily_price,
            tvl_token0_equiv AS latest_daily_total
        FROM mart_tvl_daily
        WHERE pool = %(pool)s
        ORDER BY date DESC
        LIMIT 1
    )
    SELECT
        c.pool,
        b.target_block,
        b.snapshot_block_time,
        c.current_tick AS current_tick_current,
        b.current_tick AS current_tick_block,
        c.price_token1_in_token0 AS price_current,
        b.price_token1_in_token0 AS price_block,
        d.latest_daily_date,
        d.latest_daily_price,
        d.latest_daily_total,
        c.total_tvl_token0 AS current_total,
        b.total_tvl_token0 AS block_total,
        c.active_tvl_token0 AS current_active,
        b.active_tvl_token0 AS block_active,
        c.out_of_range_tvl_token0 AS current_out_of_range,
        b.out_of_range_tvl_token0 AS block_out_of_range,
        c.total_tvl_token0 - b.total_tvl_token0 AS diff_current_vs_block_total,
        c.active_tvl_token0 - b.active_tvl_token0 AS diff_current_vs_block_active,
        c.out_of_range_tvl_token0 - b.out_of_range_tvl_token0 AS diff_current_vs_block_out_of_range,
        c.total_tvl_token0 - d.latest_daily_total AS diff_current_vs_latest_daily_total
    FROM mart_tvl_current c
    LEFT JOIN mart_tvl_block_snapshot b ON c.pool = b.pool
    LEFT JOIN latest_daily d ON c.pool = d.pool
    WHERE c.pool = %(pool)s
    '''
), {'pool': POOL})
snapshot_compare

1 rows


,c.pool,target_block,snapshot_block_time,current_tick_current,current_tick_block,price_current,price_block,latest_daily_date,latest_daily_price,latest_daily_total,current_total,block_total,current_active,block_active,current_out_of_range,block_out_of_range,diff_current_vs_block_total,diff_current_vs_block_active,diff_current_vs_block_out_of_range,diff_current_vs_latest_daily_total
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24820333,2026-04-06 11:28:11,199587,199587,2150.092004,2150.092004,2026-04-06,2150.092004,9.362351e+07,9.362351e+07,9.362351e+07,7.460444e+07,7.460444e+07,1.901907e+07,1.901907e+07,-1.490116e-08,8.940697e-08,0.0,0.0


## 3. Reconciliation checks

Ici on checke les invariants les plus importants:
- `total = active + out_of_range` sur le snapshot courant
- `total = active + out_of_range` sur le block snapshot
- pas de TVL negative

In [8]:
reconciliation_checks = query_df(dedent(
    '''
    SELECT
        source,
        pool,
        total_tvl_token0,
        active_tvl_token0,
        out_of_range_tvl_token0,
        tvl_reconciliation_gap,
        abs(tvl_reconciliation_gap) AS abs_gap,
        if(abs(tvl_reconciliation_gap) <= %(epsilon)s, 'PASS', 'CHECK') AS gap_status,
        if(total_tvl_token0 >= 0 AND active_tvl_token0 >= 0 AND out_of_range_tvl_token0 >= 0, 'PASS', 'CHECK') AS non_negative_status
    FROM (
        SELECT
            'mart_tvl_current' AS source,
            pool,
            total_tvl_token0,
            active_tvl_token0,
            out_of_range_tvl_token0,
            tvl_reconciliation_gap
        FROM mart_tvl_current
        WHERE pool = %(pool)s

        UNION ALL

        SELECT
            'mart_tvl_block_snapshot' AS source,
            pool,
            total_tvl_token0,
            active_tvl_token0,
            out_of_range_tvl_token0,
            tvl_reconciliation_gap
        FROM mart_tvl_block_snapshot
        WHERE pool = %(pool)s
    )
    ORDER BY source
    '''
), {'pool': POOL, 'epsilon': 1e-6})
reconciliation_checks

2 rows


,source,pool,total_tvl_token0,active_tvl_token0,out_of_range_tvl_token0,tvl_reconciliation_gap,abs_gap,gap_status,non_negative_status
0,mart_tvl_block_snapshot,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,9.362351e+07,7.460444e+07,1.901907e+07,8.195639e-08,8.195639e-08,PASS,PASS
1,mart_tvl_current,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,9.362351e+07,7.460444e+07,1.901907e+07,-2.235174e-08,2.235174e-08,PASS,PASS


## 4. Distribution sanity

On veut verifier que la distribution recompose bien le snapshot courant:
- somme des bins = TVL totale
- somme des `% liquidity` = 100
- un seul bin actif
- le bin actif de la distribution matche le current bin du snapshot

In [9]:
distribution_checks = query_df(dedent(
    '''
    WITH distribution_rollup AS (
        SELECT
            pool,
            countIf(is_active_bin = 1) AS active_bins,
            sum(pct_liquidity) AS pct_sum,
            sum(tvl_token0) AS distribution_total_tvl_token0,
            sumIf(tvl_token0, is_active_bin = 1) AS distribution_active_bin_tvl_token0,
            sumIf(amount0, is_active_bin = 1) AS distribution_active_bin_token0,
            sumIf(amount1, is_active_bin = 1) AS distribution_active_bin_token1
        FROM mart_tvl_distribution
        WHERE pool = %(pool)s
        GROUP BY pool
    )
    SELECT
        c.pool,
        d.active_bins,
        d.pct_sum,
        c.total_tvl_token0 AS current_total_tvl_token0,
        d.distribution_total_tvl_token0,
        c.current_bin_tvl_token0,
        d.distribution_active_bin_tvl_token0,
        c.current_bin_token0,
        d.distribution_active_bin_token0,
        c.current_bin_token1,
        d.distribution_active_bin_token1,
        c.total_tvl_token0 - d.distribution_total_tvl_token0 AS diff_total,
        c.current_bin_tvl_token0 - d.distribution_active_bin_tvl_token0 AS diff_active_bin_tvl,
        c.current_bin_token0 - d.distribution_active_bin_token0 AS diff_active_bin_token0,
        c.current_bin_token1 - d.distribution_active_bin_token1 AS diff_active_bin_token1
    FROM mart_tvl_current c
    INNER JOIN distribution_rollup d ON c.pool = d.pool
    WHERE c.pool = %(pool)s
    '''
), {'pool': POOL})
distribution_checks

1 rows


,pool,active_bins,pct_sum,current_total_tvl_token0,distribution_total_tvl_token0,current_bin_tvl_token0,distribution_active_bin_tvl_token0,current_bin_token0,distribution_active_bin_token0,current_bin_token1,distribution_active_bin_token1,diff_total,diff_active_bin_tvl,diff_active_bin_token0,diff_active_bin_token1
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,1,100.0,9.362351e+07,9.362351e+07,336558.14942,336558.14942,83112.776728,83112.776728,117.876524,117.876524,-1.490116e-08,0.0,0.0,0.0


## 5. Current bin decomposition

Le bin actif est souvent ce qu'on veut afficher dans un dashboard. On le sort proprement ici.

In [10]:
current_bin_snapshot = query_df(dedent(
    '''
    SELECT
        pool,
        current_tick,
        current_bin_tick_lower,
        current_bin_tick_upper,
        current_bin_liquidity,
        current_bin_token0,
        current_bin_token1,
        current_bin_token0_in_token0,
        current_bin_token1_in_token0,
        current_bin_tvl_token0,
        current_bin_tvl_token0 / nullIf(total_tvl_token0, 0) AS current_bin_share_of_total,
        current_bin_tvl_token0 / nullIf(active_tvl_token0, 0) AS current_bin_share_of_active
    FROM mart_tvl_current
    WHERE pool = %(pool)s
    '''
), {'pool': POOL})
current_bin_snapshot

1 rows


,pool,current_tick,current_bin_tick_lower,current_bin_tick_upper,current_bin_liquidity,current_bin_token0,current_bin_token1,current_bin_token0_in_token0,current_bin_token1_in_token0,current_bin_tvl_token0,current_bin_share_of_total,current_bin_share_of_active
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199587,199580,199590,1.451949e+19,83112.776728,117.876524,83112.776728,253445.372693,336558.14942,0.003595,0.004511


## 6. Distribution around current tick

Fenetre locale autour du tick courant pour voir si la profondeur ressemble a ce qu'on attend.

In [11]:
TICK_WINDOW = int(os.getenv('TICK_WINDOW', '400'))

local_distribution = query_df(dedent(
    '''
    WITH current_state AS (
        SELECT current_tick
        FROM mart_tvl_current
        WHERE pool = %(pool)s
    )
    SELECT
        d.pool,
        d.tick_lower,
        d.tick_upper,
        d.tick_mid,
        d.is_active_bin,
        d.liquidity,
        d.amount0,
        d.amount1,
        d.tvl_token0,
        d.pct_liquidity,
        d.price_lower,
        d.price_upper
    FROM mart_tvl_distribution d
    CROSS JOIN current_state c
    WHERE d.pool = %(pool)s
      AND d.tick_mid BETWEEN c.current_tick - %(tick_window)s AND c.current_tick + %(tick_window)s
    ORDER BY d.tick_lower
    '''
), {'pool': POOL, 'tick_window': TICK_WINDOW})
local_distribution.head(50)

80 rows


,pool,tick_lower,tick_upper,tick_mid,is_active_bin,liquidity,amount0,amount1,tvl_token0,pct_liquidity,price_lower,price_upper
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199190,199200,199195.0,0,3.943371e+18,0.000000,41.693099,89643.999011,0.099161,2237.28,2235.05
1,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199200,199210,199205.0,0,3.980449e+18,0.000000,42.106166,90532.130829,0.100094,2235.05,2232.81
2,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199210,199220,199215.0,0,3.989024e+18,0.000000,42.217982,90772.545537,0.100309,2232.81,2230.58
3,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199220,199230,199225.0,0,3.992298e+18,0.000000,42.273763,90892.479734,0.100392,2230.58,2228.35
4,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199230,199240,199235.0,0,3.997492e+18,0.000000,42.349929,91056.243830,0.100522,2228.35,2226.12
5,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199240,199250,199245.0,0,4.020439e+18,0.000000,42.614326,91624.720875,0.101099,2226.12,2223.90
6,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199250,199260,199255.0,0,4.025687e+18,0.000000,42.691298,91790.219385,0.101231,2223.90,2221.68
7,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199260,199270,199265.0,0,4.068758e+18,0.000000,43.169624,92818.663267,0.102314,2221.68,2219.46
8,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199270,199280,199275.0,0,4.074395e+18,0.000000,43.251061,92993.759553,0.102456,2219.46,2217.24
9,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199280,199290,199285.0,0,4.076089e+18,0.000000,43.290677,93078.938812,0.102499,2217.24,2215.02


In [12]:
if PLOTLY_AVAILABLE and not local_distribution.empty:
    fig = px.bar(
        local_distribution,
        x='tick_mid',
        y='tvl_token0',
        color=local_distribution['is_active_bin'].astype(str),
        title=f'TVL distribution around current tick (+/- {TICK_WINDOW}) - {POOL}',
        labels={'tick_mid': 'Tick mid', 'tvl_token0': 'TVL in token0 terms', 'color': 'is_active_bin'},
    )
    fig.update_layout(bargap=0.05)
    fig.show()
else:
    print('Plotly unavailable or local_distribution is empty.')

## 7. Daily history sanity

Ici on regarde surtout:
- que la serie journaliere est continue et non negative
- la tendance TVL
- la relation TVL / prix

In [13]:
daily_history = query_df(dedent(
    '''
    SELECT
        pool,
        date,
        tvl_token0,
        tvl_token1,
        price_token1_in_token0,
        tvl_token0_equiv,
        lagInFrame(tvl_token0_equiv, 1) OVER (PARTITION BY pool ORDER BY date) AS prev_total_tvl_token0
    FROM mart_tvl_daily
    WHERE pool = %(pool)s
    ORDER BY date
    '''
), {'pool': POOL})

daily_history['daily_tvl_change_pct'] = (
    (daily_history['tvl_token0_equiv'] / daily_history['prev_total_tvl_token0']) - 1.0
)

daily_history.tail(10)

1,798 rows


,pool,date,tvl_token0,tvl_token1,price_token1_in_token0,tvl_token0_equiv,prev_total_tvl_token0,daily_tvl_change_pct
1788,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-03-28,3.875228e+07,24470.008184,1993.846431,8.754172e+07,8.685541e+07,0.007902
1789,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-03-29,3.716840e+07,25454.760932,1983.399845,8.765537e+07,8.754172e+07,0.001298
1790,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-03-30,4.402691e+07,21976.597324,2024.662060,8.852209e+07,8.765537e+07,0.009888
1791,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-03-31,5.595445e+07,16477.762216,2105.216375,9.064371e+07,8.852209e+07,0.023967
1792,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-01,6.046524e+07,14328.674127,2139.518675,9.112170e+07,9.064371e+07,0.005273
1793,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-02,4.947897e+07,20319.366093,2056.720249,9.127022e+07,9.112170e+07,0.001630
1794,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-03,4.901457e+07,20798.287798,2053.374258,9.172124e+07,9.127022e+07,0.004942
1795,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-04,5.072714e+07,19926.915582,2064.719705,9.187064e+07,9.172124e+07,0.001629
1796,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-05,5.711747e+07,16847.466982,2109.333463,9.265439e+07,9.187064e+07,0.008531
1797,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,2026-04-06,6.247228e+07,14488.325318,2150.092004,9.362351e+07,9.265439e+07,0.010459


In [14]:
daily_quality = pd.DataFrame([
    {
        'rows': len(daily_history),
        'null_total_tvl_rows': int(daily_history['tvl_token0_equiv'].isna().sum()),
        'non_positive_total_tvl_rows': int((daily_history['tvl_token0_equiv'] <= 0).fillna(False).sum()),
        'null_price_rows': int(daily_history['price_token1_in_token0'].isna().sum()),
        'non_positive_price_rows': int((daily_history['price_token1_in_token0'] <= 0).fillna(False).sum()),
        'max_abs_daily_tvl_change_pct': float(daily_history['daily_tvl_change_pct'].abs().dropna().max()),
    }
])
daily_quality

,rows,null_total_tvl_rows,non_positive_total_tvl_rows,null_price_rows,non_positive_price_rows,max_abs_daily_tvl_change_pct
0,1798,0,0,0,0,31.502128


In [15]:
if PLOTLY_AVAILABLE and not daily_history.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=daily_history['date'],
        y=daily_history['tvl_token0_equiv'],
        mode='lines',
        name='Total TVL (token0 equiv)',
        yaxis='y1',
    ))
    fig.add_trace(go.Scatter(
        x=daily_history['date'],
        y=daily_history['price_token1_in_token0'],
        mode='lines',
        name='Price token1 in token0',
        yaxis='y2',
        opacity=0.6,
    ))
    fig.update_layout(
        title=f'Daily TVL and price - {POOL}',
        yaxis=dict(title='TVL (token0 equiv)'),
        yaxis2=dict(title='Price', overlaying='y', side='right'),
        legend=dict(orientation='h'),
    )
    fig.show()
else:
    print('Plotly unavailable or daily_history is empty.')

## 8. Quick conclusions helper

Cette cellule donne un resume tres simple de l'etat du pipeline TVL.

In [16]:
summary = pd.DataFrame([
    {
        'pool': POOL,
        'snapshot_target_block': int(snapshot_compare.iloc[0]['target_block']) if not snapshot_compare.empty and pd.notna(snapshot_compare.iloc[0]['target_block']) else None,
        'current_vs_block_total_abs_diff': float(abs(snapshot_compare.iloc[0]['diff_current_vs_block_total'])) if not snapshot_compare.empty else None,
        'current_vs_block_active_abs_diff': float(abs(snapshot_compare.iloc[0]['diff_current_vs_block_active'])) if not snapshot_compare.empty else None,
        'distribution_vs_current_total_abs_diff': float(abs(distribution_checks.iloc[0]['diff_total'])) if not distribution_checks.empty else None,
        'distribution_active_bin_abs_diff': float(abs(distribution_checks.iloc[0]['diff_active_bin_tvl'])) if not distribution_checks.empty else None,
        'current_gap_abs': float(abs(reconciliation_checks.loc[reconciliation_checks['source'] == 'mart_tvl_current', 'abs_gap'].iloc[0])) if not reconciliation_checks.empty else None,
        'block_gap_abs': float(abs(reconciliation_checks.loc[reconciliation_checks['source'] == 'mart_tvl_block_snapshot', 'abs_gap'].iloc[0])) if not reconciliation_checks.empty else None,
        'distribution_active_bins': int(distribution_checks.iloc[0]['active_bins']) if not distribution_checks.empty else None,
        'distribution_pct_sum': float(distribution_checks.iloc[0]['pct_sum']) if not distribution_checks.empty else None,
    }
])
summary

,pool,snapshot_target_block,current_vs_block_total_abs_diff,current_vs_block_active_abs_diff,distribution_vs_current_total_abs_diff,distribution_active_bin_abs_diff,current_gap_abs,block_gap_abs,distribution_active_bins,distribution_pct_sum
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,24820333,1.490116e-08,8.940697e-08,1.490116e-08,0.0,2.235174e-08,8.195639e-08,1,100.0
